# [LAB-09] 5. [PBT] 분석용 데이터 전처리 - 연습문제

## 준비작업

### 라이브러리 참조

In [1]:
from jussam import load_data
from helpers import *

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


## 📚 언더라이터의 모델 투입 전 데이터 다듬기

### 당신은 보험사의 언더라이터입니다.

가입자 1,337명의 나이·체질량지수·자녀 수·흡연 여부와 실제 청구비용(`charges`)이 담긴 데이터를 받았습니다. 지난 EDA에서 **성별(`sex`)과 거주 지역(`region`)은 청구비용과 관련이 없어 제외**하기로 결론이 났습니다.

팀장은 "모델은 내가 돌릴 테니, 그대로 집어넣을 수 있는 데이터를 만들어 달라"고 합니다. 문자를 숫자로 바꾸고, 치우친 분포를 펴고, 이상치를 다듬고, 척도를 맞추는 일이 남았습니다.

전처리를 한 단계씩 밟으면서 **각 단계에서 데이터가 실제로 어떻게 바뀌는지** 확인하세요.

### 💻 코드 작성

#### 데이터 가져오기와 타입 확정

In [2]:
origin = load_data("insurance_qtcheck")
desc = load_data("insurance_qtcheck_desc")
cat_desc = load_data("insurance_qtcheck_category_desc")

# 명목형 기술통계량 표의 컬럼명이 곧 명목형 변수 목록이다
df = my_qtcheck.set_type(origin, as_category=cat_desc.columns)

📚 보험 비용 데이터셋의 품질 검사 완료 버전 (출처: 자체 정제)
📚 보험 비용 데이터셋의 연속형 변수에 대한 기술 통계량 (출처: 자체 작업)
📚 보험 비용 데이터셋의 범주형 변수에 대한 기술 통계량 (출처: 자체 작업)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   int64   
 1   sex       1337 non-null   category
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   category
 5   region    1337 non-null   category
 6   charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.3 KB


#### 변수 유형 분류와 EDA 결과 반영

In [3]:
target = "charges"

nominal_cols = my_qtcheck.get_categorical_column_names(df)
continuous_cols = my_qtcheck.get_number_column_names(df)
continuous_cols.remove(target)

print("[EDA 반영 전] 명목형 :", nominal_cols)

# EDA에서 채택되지 않은 변수를 걷어낸다
df = df.drop(columns=['sex', 'region'])
cat_desc = cat_desc.drop(columns=['sex', 'region'])
nominal_cols = cat_desc.columns.to_list()

print("[EDA 반영 후] 명목형 :", nominal_cols)
print("           연속형 :", continuous_cols)
print("           크기   :", df.shape)

[EDA 반영 전] 명목형 : ['sex', 'smoker', 'region']
[EDA 반영 후] 명목형 : ['smoker']
           연속형 : ['age', 'bmi', 'children']
           크기   : (1337, 5)


#### 명목형 라벨링

In [4]:
df1 = my_prep.labeling(df, columns=nominal_cols)
df1.head()

smoker (2종): no=0, yes=1


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


#### 더미변수 인코딩

In [5]:
df2 = my_prep.dummies(df1, columns=nominal_cols, drop_first=True)
df2.head()

smoker (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지

컬럼 수: 5개 -> 5개


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


#### 로그 변환 대상 선정

In [6]:
# 기술통계량 표가 왜도·첨도·최솟값을 근거로 변환 종류를 미리 판정해 둔다
log_cols = desc.loc[desc['log_need'] == 'log'].index.tolist()
log1p_cols = desc.loc[desc['log_need'] == 'log1p'].index.tolist()
reflect_cols = desc.loc[desc['log_need'] == 'reverse_log1p'].index.tolist()

print("순수 log  :", log_cols)
print("log(1+x)  :", log1p_cols)
print("반사 후 log:", reflect_cols)

desc[['skew', 'kurt', 'min', 'log_need']]

순수 log  : ['charges']
log(1+x)  : ['children']
반사 후 log: []


,skew,kurt,min,log_need
age,0.055,-1.244,18.000,none
bmi,0.284,-0.053,15.960,none
children,0.937,0.201,0.000,log1p
charges,1.515,1.604,1121.874,log


#### 로그 변환

In [7]:
df3 = my_prep.log_transform(df2, log_columns=log_cols,
                            log1p_columns=log1p_cols,
                            reflect_columns=reflect_cols)

컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
charges   우측 꼬리     log(x)                exp(y)                   1.51 -> -0.09
children  우측 꼬리     log(1+x)              exp(y)-1                  0.94 -> 0.26


#### 이상치 대체

In [8]:
df4 = my_prep.replace_outlier(df3, columns=continuous_cols)

이상치 대체 방식: 'bound' (기준: IQR x 1.5)
컬럼                               정상 범위       이상치                   대체값
--------------------------------------------------------------------
age                    -9.00 ~ 87.00    0개(0.0%)      -9.00 또는 87.00
bmi                    13.67 ~ 47.32    9개(0.7%)      13.67 또는 47.32
children                -1.65 ~ 2.75    0개(0.0%)       -1.65 또는 2.75


#### 다중공선성 제거

In [9]:
df5 = my_prep.reduce_vif(df4, columns=continuous_cols, threshold=10.0)


완료! 남은 변수: ['age', 'bmi', 'children']
최대 VIF = 1.01


#### 정규화

In [10]:
df6 = my_prep.scaling(df5, columns=continuous_cols, method='standard')

df6.describe().T[['mean', 'std', 'min', 'max']]

StandardScaler 적용 (3개 컬럼)
컬럼                            변환 전                  변환 후
--------------------------------------------------------
age                18.00 ~ 64.00          -1.51 ~ 1.76
bmi                15.96 ~ 47.32          -2.43 ~ 2.75
children             0.00 ~ 1.79          -1.04 ~ 2.17


,mean,std,min,max
age,-0.000,1.000,-1.512,1.765
bmi,-0.000,1.000,-2.425,2.751
children,-0.000,1.000,-1.043,2.167
smoker,0.205,0.404,0.000,1.000
charges,9.100,0.919,7.023,11.063


### 문제 풀이

#### 1. EDA에서 채택되지 않은 변수를 걷어내고 나면, 문자로 되어 있어 숫자로 바꿔 주어야 하는 변수는 단 하나만 남습니다. 그 변수는 무엇인가요?

- **정답**: `smoker`
- **복습 개념**: 명목형 변수의 확정입니다. 회귀모델은 문자를 그대로 먹지 못하므로, 먼저 어떤 변수가 명목형인지 목록을 확정해야 이후 라벨링·인코딩 대상이 정해집니다.
- **풀이 접근**: 품질 점검을 마친 데이터의 타입을 확정한 뒤 명목형 변수 목록을 뽑고, EDA에서 제외하기로 한 변수를 데이터와 목록 양쪽에서 함께 걷어냅니다.
- **근거(계산 결과)**: 처음 명목형은 `sex`·`smoker`·`region` 세 개였고, `sex`와 `region`을 제외하면 **smoker** 하나가 남습니다. 데이터도 7열에서 5열로 줄어듭니다.
- **자주 하는 실수**: 데이터프레임에서만 컬럼을 지우고 명목형 **목록**을 그대로 두면, 다음 단계에서 이미 없는 컬럼을 라벨링하려다 오류가 납니다. 데이터와 목록을 항상 같이 갱신하세요.

#### 2. 흡연 여부를 정수로 바꾸었습니다. 흡연자(`yes`)에게 부여된 정수는 얼마인가요?

- **정답**: `1`
- **복습 개념**: 라벨 인코딩입니다. 범주를 **알파벳 순서**로 0부터 차례대로 매깁니다. 순서를 직접 지정하지 않는 한 이 규칙이 그대로 적용됩니다.
- **풀이 접근**: 명목형 목록을 넘겨 라벨링을 수행하면, 컬럼별로 어떤 값이 어떤 정수가 되었는지 매핑이 출력됩니다.
- **근거(계산 결과)**: `no`와 `yes` 중 알파벳이 앞서는 `no`가 0, `yes`가 **1** 입니다.
- **실무 포인트**: 이 매핑을 확인해 두지 않으면 나중에 회귀계수의 부호를 거꾸로 읽게 됩니다. `smoker`의 계수가 양수라면 "1(흡연자)일수록 청구비용이 높다"로 읽어야 하는데, 0이 흡연자였다면 정반대 결론이 나옵니다.

#### 3. 라벨링한 명목형 변수에 더미 인코딩을 수행했습니다. 인코딩을 마친 데이터의 컬럼 수는 몇 개인가요?

- **정답**: `5`
- **복습 개념**: 더미 인코딩과 이진 변수의 예외 처리입니다. 범주가 k개면 보통 k−1개의 더미 컬럼이 생기지만, 범주가 2개뿐이면 이미 0/1이라 **더미를 만들 필요가 없습니다.**
- **풀이 접근**: 라벨링된 데이터에 명목형 목록을 넘겨 더미 인코딩을 수행하고, 실행 결과에 찍히는 컬럼 수의 변화를 확인합니다.
- **근거(계산 결과)**: 출력에 `smoker (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지` 가 찍히고 `컬럼 수: 5개 -> 5개` 로 그대로입니다. 컬럼은 `age`·`bmi`·`children`·`smoker`·`charges` **5개** 입니다.
- **헷갈리기 쉬운 점**: "인코딩을 했으니 컬럼이 늘었겠지"라고 생각하기 쉽습니다. 2범주 변수에 굳이 더미를 만들면 `smoker_1` 이 원래의 `smoker` 와 완전히 같은 값이 되어 이름만 바뀔 뿐입니다. 컬럼이 늘어나는 것은 범주가 3개 이상일 때입니다.

#### 4. 치우친 분포를 펴기 위해 로그 변환 대상으로 뽑힌 변수는 두 개입니다. 그 중 0이 들어 있어서 그냥 로그를 취할 수 없고 1을 더한 뒤 로그를 취해야 하는 변수는 무엇인가요?

- **정답**: `children`
- **복습 개념**: `log` 와 `log1p` 의 구분입니다. `log(0)` 은 −∞ 라서 계산이 깨지므로, 0이 포함된 변수는 `log(1+x)` 로 우회합니다. 갈림길을 정하는 것은 분포의 모양이 아니라 **최솟값**입니다.
- **풀이 접근**: 기술통계량 표의 판정 결과에서 순수 로그 대상과 `1+x` 로그 대상을 각각 뽑아 봅니다. 각 변수의 최솟값을 함께 보면 왜 그렇게 갈렸는지 확인할 수 있습니다.
- **근거(계산 결과)**: 왜도가 큰 변수는 `charges`(1.515)와 `children`(0.937) 둘입니다. `charges` 는 최솟값이 1,121.874로 0보다 크니 순수 `log`, **children** 은 자녀가 없는 가입자가 573명이라 최솟값이 0이므로 `log(1+x)` 로 갈립니다.
- **함께 생각해 볼 점**: `age`(왜도 0.055)와 `bmi`(0.284)는 대상에서 빠졌습니다. 이미 충분히 대칭이라 로그를 씌우면 오히려 왼쪽으로 기울어집니다. **모든 변수에 로그를 씌우는 것이 아니라, 치우친 변수만 골라 씌우는 것**입니다.

#### 5. 종속변수인 청구비용은 오른쪽으로 길게 늘어진 분포였습니다. 로그 변환을 마친 뒤 이 변수의 왜도는 얼마가 되었나요? (소수 둘째 자리)

- **정답**: `-0.09`
- **복습 개념**: 로그 변환의 효과 확인입니다. 왜도의 절댓값이 0.5 아래로 내려오면 대칭에 가까워졌다고 봅니다. 변환은 목적이 아니라 수단이므로, 씌운 뒤 실제로 펴졌는지 확인해야 합니다.
- **풀이 접근**: 선정된 세 목록(순수 로그 · `1+x` 로그 · 반사 후 로그)을 한 번에 넘겨 변환을 수행하면, 컬럼별 변환식과 함께 왜도가 어떻게 바뀌었는지 출력됩니다.
- **근거(계산 결과)**: `charges` 의 왜도가 1.51 → **−0.09** 로, 오른쪽 꼬리가 거의 완전히 펴졌습니다. `children` 도 0.94 → 0.26으로 함께 내려왔습니다.
- **실무 포인트**: 부호가 음수로 바뀐 것에 놀랄 필요는 없습니다. −0.09는 절댓값이 0.1도 되지 않아 사실상 대칭입니다. 다만 이제 `charges` 는 달러가 아니라 **로그 스케일**이므로, 나중에 예측값을 실제 금액으로 읽으려면 지수를 취해 되돌려야 합니다.

#### 6. 연속형 변수들의 이상치를 경계값으로 대체했습니다. 이때 대체된 값은 모두 몇 개인가요?

- **정답**: `9`
- **복습 개념**: IQR 기준(1.5배)의 이상치 판정과 경계값 대체입니다. 행을 지우지 않고 경계값으로 눌러 담으므로 **표본 수는 그대로 유지**됩니다.
- **풀이 접근**: 연속형 독립변수 목록을 넘겨 이상치 대체를 수행하면, 컬럼별로 정상 범위와 이상치 개수가 표로 출력됩니다. 그 개수를 모두 더합니다.
- **근거(계산 결과)**: `age` 0개, `bmi` **9개**(0.7%), `children` 0개로 합계 **9개** 입니다. 체질량지수가 47.32를 넘는 9명이 경계값으로 눌렸습니다.
- **헷갈리기 쉬운 점**: `children` 의 정상 범위가 −1.65 ~ 2.75 라는 이상한 숫자로 나옵니다. 자녀 수가 음수일 리 없는데도 이렇게 나오는 것은, 바로 앞 단계에서 이미 `log(1+x)` 로 변환된 값을 기준으로 범위를 계산했기 때문입니다. **전처리 순서가 결과를 바꾼다**는 것을 보여주는 대목입니다.

#### 7. 마지막으로 정규화까지 마쳤습니다. 그런데 데이터에는 평균이 0으로 맞춰지지 않은 채 남아 있는 변수가 하나 있습니다. 그 변수는 무엇인가요?

- **정답**: `charges`
- **복습 개념**: 정규화 대상의 범위입니다. 표준화는 **독립변수의 척도를 맞추는** 작업이므로, 종속변수는 대상에 넣지 않습니다. 예측 대상까지 표준화하면 예측값을 원래 단위로 되돌리는 일이 번거로워집니다.
- **풀이 접근**: 연속형 독립변수 목록을 넘겨 표준화를 수행한 뒤, 전체 컬럼의 평균과 표준편차를 확인합니다. 평균이 0이 아닌 컬럼을 찾습니다.
- **근거(계산 결과)**: `age`·`bmi`·`children` 세 개만 평균 0, 표준편차 1로 맞춰졌고, **charges** 는 평균 9.100 · 표준편차 0.919 로 로그 스케일 그대로 남아 있습니다. (`smoker` 도 0/1 이진값이라 대상이 아닙니다.)
- **결론**: 이제 팀장에게 넘길 데이터가 완성되었습니다. 함께 전해야 할 말은 두 가지입니다 — **첫째, `charges` 는 로그 스케일이므로 예측값을 달러로 읽으려면 지수를 취해 되돌려야 합니다. 둘째, `age`·`bmi`·`children` 은 표준화되어 있어 계수를 "1살 늘 때"가 아니라 "1 표준편차만큼 늘 때"로 읽어야 합니다.** 참고로 바로 앞 단계에서 다중공선성도 점검했는데 최대 VIF가 1.01로 임계값 10에 한참 못 미쳐 제거된 변수는 없었습니다.